# 01 — Exploração da camada silver

Pré-requisito: `make ingest preprocess`. A lógica de produção fica em `src/`; este notebook só explora.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
plt.rcParams["figure.dpi"] = 120
SILVER = ROOT / "data" / "silver"
sns.set_theme(style="whitegrid", palette="pastel")

## Ementas por tipo e mês

In [ ]:
propositions = pl.read_parquet(SILVER / "propositions.parquet")
monthly = propositions.group_by("period", "type_code").len().sort("period").to_pandas()
fig, ax = plt.subplots(figsize=(10, 4))
sns.lineplot(data=monthly, x="period", y="len", hue="type_code", ax=ax)
ax.set_title("Proposições por mês e tipo")
ax.set_xlabel("Mês de apresentação")
ax.set_ylabel("Proposições")
plt.tight_layout()

## Tamanho dos textos (tokens)

In [ ]:
speeches = pl.read_parquet(SILVER / "speeches.parquet")
lengths = pl.concat(
    [
        propositions.select(
            pl.col("text_clean").str.count_matches(r"\S+").alias("tokens")
        ).with_columns(pl.lit("ementas").alias("corpus")),
        speeches.select(
            pl.col("text_clean").str.count_matches(r"\S+").alias("tokens")
        ).with_columns(pl.lit("discursos").alias("corpus")),
    ]
).to_pandas()
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(data=lengths, x="tokens", hue="corpus", log_scale=True, element="step", ax=ax)
ax.set_title("Distribuição do tamanho dos textos")
ax.set_xlabel("Tokens (escala log)")
ax.set_ylabel("Documentos")
plt.tight_layout()